In [0]:
import io
import unicodedata
import re
import logging
import pandas as pd

from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from pyspark.sql.functions import current_timestamp, lit

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ── Google Drive service account credentials ─────────────────────────────────
# Stored as individual secrets (one per JSON field), since Databricks secrets
# are plain string values and can't hold a full JSON blob.
_SECRET_SCOPE = "ss2-bronze-layer"

GOOGLE_SERVICE_ACCOUNT_INFO = {
    "type": dbutils.secrets.get(scope=_SECRET_SCOPE, key="gsa_type"),
    "project_id": dbutils.secrets.get(scope=_SECRET_SCOPE, key="gsa_project_id"),
    "private_key_id": dbutils.secrets.get(scope=_SECRET_SCOPE, key="gsa_private_key_id"),
    "private_key": dbutils.secrets.get(scope=_SECRET_SCOPE, key="gsa_private_key").replace("\\n", "\n"),
    "client_email": dbutils.secrets.get(scope=_SECRET_SCOPE, key="gsa_client_email"),
    "client_id": dbutils.secrets.get(scope=_SECRET_SCOPE, key="gsa_client_id"),
    "auth_uri": dbutils.secrets.get(scope=_SECRET_SCOPE, key="gsa_auth_uri"),
    "token_uri": dbutils.secrets.get(scope=_SECRET_SCOPE, key="gsa_token_uri"),
    "auth_provider_x509_cert_url": dbutils.secrets.get(scope=_SECRET_SCOPE, key="gsa_auth_provider_x509_cert_url"),
    "client_x509_cert_url": dbutils.secrets.get(scope=_SECRET_SCOPE, key="gsa_client_x509_cert_url"),
    "universe_domain": dbutils.secrets.get(scope=_SECRET_SCOPE, key="gsa_universe_domain"),
}

# ── Pipeline configuration ────────────────────────────────────────────────────
WRITE_MODE      = "overwrite"   # "overwrite" or "append"
TARGET_SCHEMA   = "sandbox_bronze_ss2"  # must match the schema created in schema_creation.py
SOURCE_SYSTEM   = "WHO Mortality Database Portal (Google Drive)"  # lineage tag appended to every ingested row

# WHO export format: number of metadata lines before the real header row
# (Region Code, Region Name, Country Code, Country Name, Export date,
# Source location, "Metadata information:", blank line -> 8 lines)
METADATA_LINES = 8

# Each entry is an INDEPENDENT source -> 1 bronze table each, no relationships.
GDRIVE_FILES = [
    # {
    #     "country": "guatemala",
    #     "file_id": "1WRWauPj5DKr4jZg_mBDvCvPlGC0DkQb7",
    #     "file_name": "who_mortality_db_Guatemala.csv",
    # },
    {
        "country": "costa_rica",
        "file_id": "1xh8E2--ZSe_9gMjpCjrjD4dHQkhI4hJE",
        "file_name": "who_mortality_db_CostaRica.csv",
    },
]


# ── Helpers ───────────────────────────────────────────────────────────────────

def clean_name(name):
    # Normalize table/column names to snake_case; strips .csv extension if present
    name = name.lower().replace(".csv", "")
    # ñ must be transliterated before NFKD decomposition (ñ → ni, so "año" → "anio")
    name = name.replace("ñ", "ni")
    # Strip diacritical marks from accented vowels (á→a, é→e, ó→o, ú→u, etc.)
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r'[^a-z0-9_]', '_', name)
    name = re.sub(r'_+', '_', name)
    return name.strip('_')


def dedupe_columns(cols):
    # After normalization, distinct source columns can collide (e.g. "CIE-10" and "CIE_10" → "cie_10")
    seen = {}
    result = []
    for col in cols:
        count = seen.get(col, 0)
        seen[col] = count + 1
        result.append(col if count == 0 else f"{col}_{count + 1}")
    return result


def get_drive_service(service_account_info):
    scopes = ["https://www.googleapis.com/auth/drive.readonly"]
    creds = Credentials.from_service_account_info(service_account_info, scopes=scopes)
    return build("drive", "v3", credentials=creds)


def download_drive_csv(service, file_id, file_name):
    # Confirm the file exists and is accessible before downloading content
    metadata = service.files().get(fileId=file_id, fields="id, name, mimeType, size").execute()
    logger.info(f"    File found in Drive: {metadata.get('name')} "
                 f"({metadata.get('mimeType')}, {metadata.get('size', '?')} bytes)")

    request = service.files().get_media(fileId=file_id)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)

    done = False
    while not done:
        status, done = downloader.next_chunk()
        if status:
            logger.info(f"    Download: {int(status.progress() * 100)}%")

    buffer.seek(0)
    return buffer


def read_who_csv(buffer, metadata_lines):
    # WHO export CSVs have `metadata_lines` lines of metadata before the real
    # header row; skip them so pandas parses the actual data header.
    try:
        df = pd.read_csv(buffer, skiprows=metadata_lines, sep=None, engine="python")
    except Exception:
        # Fallback for files with malformed quoting (e.g. unescaped quotes mid-field)
        buffer.seek(0)
        df = pd.read_csv(buffer, skiprows=metadata_lines, sep=None, engine="python", on_bad_lines="skip")

    # Log raw column names so schema drift across files is immediately visible in the job log
    logger.info(f"    Raw schema ({len(df.columns)} cols): {list(df.columns)}")
    # Normalize HERE so downstream concatenation/merging aligns on clean names.
    df.columns = dedupe_columns([clean_name(c) for c in df.columns])
    # Drop columns that pandas auto-generates from trailing delimiters (e.g. "Unnamed: 8")
    df = df.loc[:, ~df.columns.str.match(r'^unnamed_\d+$')]
    logger.info(f"    Clean schema ({len(df.columns)} cols): {list(df.columns)}")
    return df


def ingest_gdrive_file(service, country, file_id, file_name, schema, write_mode, source_system, metadata_lines):
    table_name = f"{schema}.who_{clean_name(country)}"
    logger.info("-" * 60)
    logger.info(f"Source file  : {file_name} ({country})")
    logger.info(f"Target table : {table_name}  [mode={write_mode}]")

    try:
        logger.info(f"  Downloading: {file_name}")
        buffer = download_drive_csv(service, file_id, file_name)
        df = read_who_csv(buffer, metadata_lines)
    except Exception as exc:
        # Log and skip so one bad source doesn't abort the entire pipeline
        logger.warning(f"  Skipped {file_name}: {exc}")
        return

    if df.empty:
        logger.warning(f"  No valid data for '{country}', skipping write.")
        return

    # Resolve object columns with mixed types (int rows + str rows) that Arrow cannot infer.
    # errors='coerce' turns non-numeric values into NaN so we can measure the numeric ratio.
    for col in df.select_dtypes(include="object").columns:
        coerced = pd.to_numeric(df[col], errors="coerce")
        non_null = df[col].notna().sum()
        if non_null > 0 and coerced.notna().sum() / non_null >= 0.9:
            # Column is predominantly numeric → keep as float (NaN-compatible)
            df[col] = coerced
        else:
            # Text column → cast non-null values to str so Arrow maps to StringType uniformly
            df[col] = df[col].where(df[col].isna(), df[col].astype(str))

    sdf = spark.createDataFrame(df)
    sdf = (sdf
           .withColumn("ingestion_timestamp", current_timestamp())
           .withColumn("source_system", lit(source_system))
           .withColumn("source_file", lit(file_name))
           .withColumn("gdrive_file_id", lit(file_id))
           .withColumn("country", lit(country)))

    (sdf.write
        .format("delta")
        .mode(write_mode)
        .option("mergeSchema", "true")   # allows new columns across runs without schema errors
        .saveAsTable(table_name))

    logger.info(f"  Wrote {len(df):,} rows → {table_name}")


# ── Entry point ───────────────────────────────────────────────────────────────
try:
    drive_service = get_drive_service(GOOGLE_SERVICE_ACCOUNT_INFO)

    for config in GDRIVE_FILES:
        ingest_gdrive_file(
            service=drive_service,
            country=config["country"],
            file_id=config["file_id"],
            file_name=config["file_name"],
            schema=TARGET_SCHEMA,
            write_mode=WRITE_MODE,
            source_system=SOURCE_SYSTEM,
            metadata_lines=METADATA_LINES,
        )

    logger.info("Pipeline completed successfully.")
except Exception as e:
    logger.error(f"Pipeline failed: {e}")
    raise  # re-raise so the Databricks job status reflects the failure
